In [1]:
import requests
from config import (POP_FLOW_BASE_URL,POP_FLOW_SERVICE_KEY, START_DATE, END_DATE, BATCH_MONTHS)
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
def read_districts_csv(path):
    df = pd.read_csv(path)

    required = {"Districts","Code"}

    if not required.issubset(df.columns):
        raise ValueError("CSV missing required columns.")

    df["Districts"] = df["Districts"].str.strip()
    df["Code"] = pd.to_numeric(df["Code"], errors="raise")

    return dict(zip(df['Districts'],df['Code']))


In [3]:

districts = read_districts_csv('/workspaces/korea-real-estate-population-movement/data/seoul_district_codes.csv')
print(len(districts))

end_date = 202303

param = {
    "serviceKey": POP_FLOW_SERVICE_KEY,
    "mvinAdmmCd": districts["종로구"],
    "mvtAdmmCd": districts["강북구"],
    "srchFrYm": START_DATE,
    "srchToYm": end_date,
    "lv": 2,
    "type":"XML",
    "numOfRows":3,
    "pageNo":1
}

response = requests.get(POP_FLOW_BASE_URL,params=param)
root = ET.fromstring(response.text)
print(ET.tostring(root,encoding="unicode"))

25
<Response>
    <head>
        <resultCode>0</resultCode>
        <resultMsg>NORMAL_SERVICE</resultMsg>
        <totalCount>3</totalCount>
        <numOfRows>3</numOfRows>
        <pageNo>1</pageNo>
    </head>
    <items>
        <item>
            <statsYm>202301</statsYm>
            <mvinAdmmCd>1111000000</mvinAdmmCd>
            <mvtAdmmCd>1130500000</mvtAdmmCd>
            <mvinCtpvNm>서울특별시</mvinCtpvNm>
            <mvtCtpvNm>서울특별시</mvtCtpvNm>
            <mvinSggNm>종로구</mvinSggNm>
            <mvtSggNm>강북구</mvtSggNm>
            <totNmprCnt>9</totNmprCnt>
            <maleNmprCnt>6</maleNmprCnt>
            <femlNmprCnt>3</femlNmprCnt>
            <male0AgeNmprCnt>0</male0AgeNmprCnt>
            <male1AgeNmprCnt>0</male1AgeNmprCnt>
            <male2AgeNmprCnt>0</male2AgeNmprCnt>
            <male3AgeNmprCnt>0</male3AgeNmprCnt>
            <male4AgeNmprCnt>0</male4AgeNmprCnt>
            <male5AgeNmprCnt>0</male5AgeNmprCnt>
            <male6AgeNmprCnt>0</male6AgeNmprCnt>
    

In [4]:
items = root.findall(".//item")

data = []
columns_wanted = ["statsYm",   
    "mvinCtpvNm",
    "mvinSggNm",
    "mvtCtpvNm",
    "mvtSggNm",
    "totNmprCnt",
    "maleNmprCnt",
    "femlNmprCnt",
    
]


for item in items:
    row = {}

    for child in item:
        if child.tag in columns_wanted:
            row[child.tag] = child.text

    data.append(row)

df = pd.DataFrame(data)
df["statsYm"].unique()

<StringArray>
['202301', '202302', '202303']
Length: 3, dtype: str

In [5]:
df = df.rename(columns={
    "statsYm":"date",
    "mvinCtpvNm":"from_province",
    "mvtCtpvNm":"to_province",
    "mvinSggNm":"from_district",
    "mvtSggNm":"to_district",
    "totNmprCnt":"total_people",
    "maleNmprCnt":"male",
    "femlNmprCnt":"female"})

df


,date,from_province,to_province,from_district,to_district,total_people,male,female
0,202301,서울특별시,서울특별시,종로구,강북구,9,6,3
1,202302,서울특별시,서울특별시,종로구,강북구,37,14,23
2,202303,서울특별시,서울특별시,종로구,강북구,17,10,7
